In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

RISK_FIGURE_DIR = (
    PROJECT_ROOT
    / "reports"
    / "figures"
    / "risk_analysis"
)

RISK_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

EVALUATION_PANEL_FILE = (
    PROCESSED_DATA_DIR
    / "19_strategy_benchmark_evaluation_panel_2015_2025.parquet"
)

print("Risk-analysis environment was initialized successfully.")

In [ ]:
evaluation_panel_df = pd.read_parquet(
    EVALUATION_PANEL_FILE
)

evaluation_panel_df["return_month"] = pd.to_datetime(
    evaluation_panel_df["return_month"]
)

risk_series_mapping = {
    "quality_top_quintile_net":
        "Quality Top Quintile",

    "six_factor_top_quintile_net":
        "Six-Factor Top Quintile",

    "crsp_value_weighted_total_return":
        "CRSP S&P 500 Value Weighted",

    "crsp_equal_weighted_total_return":
        "CRSP S&P 500 Equal Weighted"
}

required_columns = (
    ["return_month", "rf"]
    + list(risk_series_mapping.keys())
)

missing_columns = [
    column
    for column in required_columns
    if column not in evaluation_panel_df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

risk_return_df = (
    evaluation_panel_df[
        required_columns
    ]
    .rename(columns=risk_series_mapping)
    .sort_values("return_month")
    .reset_index(drop=True)
)

duplicate_months = (
    risk_return_df["return_month"]
    .duplicated()
    .sum()
)

if duplicate_months != 0:
    raise ValueError(
        "Duplicate months were found."
    )

print("Risk-analysis data was loaded successfully.")
print("Number of months:", len(risk_return_df))
print(
    "Date range:",
    risk_return_df["return_month"].min().date(),
    "to",
    risk_return_df["return_month"].max().date()
)

print("\nMissing values:")
print(risk_return_df.isna().sum())

In [ ]:
risk_series_names = list(
    risk_series_mapping.values()
)


def calculate_tail_risk_summary(
    data,
    return_column
):
    sample = (
        data[
            ["return_month", return_column, "rf"]
        ]
        .dropna()
        .copy()
    )

    returns = sample[return_column].astype(float)
    excess_returns = (
        returns
        - sample["rf"].astype(float)
    )

    wealth = (
        1.0 + returns
    ).cumprod()

    drawdown = (
        wealth
        / wealth.cummax()
        - 1.0
    )

    downside_excess_returns = np.minimum(
        excess_returns,
        0.0
    )

    annualized_downside_deviation = (
        np.sqrt(
            np.mean(
                downside_excess_returns ** 2
            )
        )
        * np.sqrt(12.0)
    )

    percentile_5 = returns.quantile(0.05)
    percentile_1 = returns.quantile(0.01)

    cvar_95_sample = returns[
        returns <= percentile_5
    ]

    cvar_99_sample = returns[
        returns <= percentile_1
    ]

    return {
        "series":
            return_column,

        "number_of_months":
            len(returns),

        "annualized_volatility":
            returns.std(ddof=1) * np.sqrt(12.0),

        "annualized_downside_deviation":
            annualized_downside_deviation,

        "historical_var_95":
            -percentile_5,

        "historical_cvar_95":
            -cvar_95_sample.mean(),

        "historical_var_99":
            -percentile_1,

        "historical_cvar_99":
            -cvar_99_sample.mean(),

        "maximum_drawdown":
            drawdown.min(),

        "worst_month":
            returns.min(),

        "loss_month_rate":
            (returns < 0).mean(),

        "skewness":
            returns.skew(),

        "excess_kurtosis":
            returns.kurt()
    }


tail_risk_records = []

for series_name in risk_series_names:
    tail_risk_records.append(
        calculate_tail_risk_summary(
            risk_return_df,
            series_name
        )
    )

tail_risk_summary_df = (
    pd.DataFrame(tail_risk_records)
    .set_index("series")
)

print("Tail-risk summary:")
display(
    tail_risk_summary_df.round(4)
)

In [ ]:
def calculate_month_difference(
    start_date,
    end_date
):
    return (
        (end_date.year - start_date.year) * 12
        + end_date.month
        - start_date.month
    )


def identify_maximum_drawdown_episode(
    data,
    return_column
):
    sample = (
        data[
            ["return_month", return_column]
        ]
        .dropna()
        .set_index("return_month")
        .sort_index()
    )

    returns = sample[return_column].astype(float)

    wealth = (
        1.0 + returns
    ).cumprod()

    running_peak = wealth.cummax()

    drawdown = (
        wealth
        / running_peak
        - 1.0
    )

    trough_date = drawdown.idxmin()

    peak_date = (
        wealth.loc[:trough_date]
        .idxmax()
    )

    peak_wealth = wealth.loc[peak_date]

    post_trough_wealth = wealth.loc[
        wealth.index > trough_date
    ]

    recovery_candidates = post_trough_wealth[
        post_trough_wealth >= peak_wealth
    ]

    if len(recovery_candidates) > 0:
        recovery_date = (
            recovery_candidates.index[0]
        )

        recovery_status = "Recovered"

        peak_to_recovery_months = (
            calculate_month_difference(
                peak_date,
                recovery_date
            )
        )
    else:
        recovery_date = pd.NaT
        recovery_status = "Not recovered"

        peak_to_recovery_months = np.nan

    peak_to_trough_months = (
        calculate_month_difference(
            peak_date,
            trough_date
        )
    )

    return {
        "series":
            return_column,

        "maximum_drawdown":
            drawdown.loc[trough_date],

        "peak_date":
            peak_date,

        "trough_date":
            trough_date,

        "recovery_date":
            recovery_date,

        "recovery_status":
            recovery_status,

        "peak_to_trough_months":
            peak_to_trough_months,

        "peak_to_recovery_months":
            peak_to_recovery_months
    }


drawdown_episode_records = []

for series_name in risk_series_names:
    drawdown_episode_records.append(
        identify_maximum_drawdown_episode(
            risk_return_df,
            series_name
        )
    )

drawdown_episode_summary_df = (
    pd.DataFrame(
        drawdown_episode_records
    )
    .set_index("series")
)

print("Maximum-drawdown episode summary:")
display(drawdown_episode_summary_df)

In [ ]:
stress_periods = {
    "2015-2016 Market Selloff": (
        pd.Timestamp("2015-06-30"),
        pd.Timestamp("2016-02-29")
    ),

    "2018 Fourth-Quarter Selloff": (
        pd.Timestamp("2018-10-31"),
        pd.Timestamp("2018-12-31")
    ),

    "2020 COVID-19 Crash": (
        pd.Timestamp("2020-02-29"),
        pd.Timestamp("2020-03-31")
    ),

    "2022 Market Drawdown": (
        pd.Timestamp("2022-01-31"),
        pd.Timestamp("2022-10-31")
    )
}


def calculate_stress_period_metrics(
    data,
    return_column,
    period_name,
    start_date,
    end_date
):
    period_sample = (
        data.loc[
            (
                data["return_month"]
                >= start_date
            )
            & (
                data["return_month"]
                <= end_date
            ),
            [
                "return_month",
                return_column
            ]
        ]
        .dropna()
        .copy()
    )

    returns = (
        period_sample[return_column]
        .astype(float)
    )

    if len(returns) == 0:
        raise ValueError(
            f"No observations were found for {period_name}."
        )

    cumulative_return = (
        (1.0 + returns).prod()
        - 1.0
    )

    wealth_values = np.concatenate([
        np.array([1.0]),
        (1.0 + returns).cumprod().to_numpy()
    ])

    running_peak_values = np.maximum.accumulate(
        wealth_values
    )

    drawdown_values = (
        wealth_values
        / running_peak_values
        - 1.0
    )

    return {
        "stress_period":
            period_name,

        "series":
            return_column,

        "start_date":
            start_date,

        "end_date":
            end_date,

        "number_of_months":
            len(returns),

        "cumulative_return":
            cumulative_return,

        "worst_monthly_return":
            returns.min(),

        "average_monthly_return":
            returns.mean(),

        "period_maximum_drawdown":
            drawdown_values.min()
    }


stress_test_records = []

for period_name, dates in stress_periods.items():
    start_date, end_date = dates

    for series_name in risk_series_names:
        stress_test_records.append(
            calculate_stress_period_metrics(
                data=risk_return_df,
                return_column=series_name,
                period_name=period_name,
                start_date=start_date,
                end_date=end_date
            )
        )

stress_test_summary_df = pd.DataFrame(
    stress_test_records
)

stress_return_pivot_df = (
    stress_test_summary_df
    .pivot(
        index="stress_period",
        columns="series",
        values="cumulative_return"
    )
)

stress_drawdown_pivot_df = (
    stress_test_summary_df
    .pivot(
        index="stress_period",
        columns="series",
        values="period_maximum_drawdown"
    )
)

print("Stress-period cumulative returns:")
display(
    stress_return_pivot_df.round(4)
)

print("Stress-period maximum drawdowns:")
display(
    stress_drawdown_pivot_df.round(4)
)

In [ ]:
TAIL_RISK_FILE = (
    PROCESSED_DATA_DIR
    / "25_tail_risk_summary_2015_2025.csv"
)

DRAWDOWN_EPISODE_FILE = (
    PROCESSED_DATA_DIR
    / "26_maximum_drawdown_episodes_2015_2025.csv"
)

STRESS_TEST_FILE = (
    PROCESSED_DATA_DIR
    / "27_stress_period_analysis_2015_2025.csv"
)

tail_risk_summary_df.to_csv(
    TAIL_RISK_FILE
)

drawdown_episode_summary_df.to_csv(
    DRAWDOWN_EPISODE_FILE
)

stress_test_summary_df.to_csv(
    STRESS_TEST_FILE,
    index=False
)

print("Static risk-analysis files were saved successfully.")

In [ ]:
rolling_risk_window = 36

risk_indexed_df = (
    risk_return_df
    .set_index("return_month")
    .sort_index()
)

rolling_volatility_df = (
    risk_indexed_df[risk_series_names]
    .rolling(
        window=rolling_risk_window,
        min_periods=rolling_risk_window
    )
    .std(ddof=1)
    * np.sqrt(12.0)
)

excess_return_df = (
    risk_indexed_df[risk_series_names]
    .sub(
        risk_indexed_df["rf"],
        axis=0
    )
)


def rolling_downside_deviation(values):
    downside_values = np.minimum(
        values,
        0.0
    )

    return (
        np.sqrt(
            np.mean(
                downside_values ** 2
            )
        )
        * np.sqrt(12.0)
    )


rolling_downside_deviation_df = (
    excess_return_df
    .rolling(
        window=rolling_risk_window,
        min_periods=rolling_risk_window
    )
    .apply(
        rolling_downside_deviation,
        raw=True
    )
)

risk_color_mapping = {
    "Quality Top Quintile":
        "#1f77b4",

    "Six-Factor Top Quintile":
        "#d62728",

    "CRSP S&P 500 Value Weighted":
        "#2ca02c",

    "CRSP S&P 500 Equal Weighted":
        "#7f7f7f"
}

fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(12, 11),
    sharex=True
)

plot_dates = (
    rolling_volatility_df
    .index
    .to_numpy()
)

for series_name in risk_series_names:
    axes[0].plot(
        plot_dates,
        rolling_volatility_df[
            series_name
        ].to_numpy(dtype=float),
        label=series_name,
        color=risk_color_mapping[series_name],
        linewidth=2.0
    )

    axes[1].plot(
        plot_dates,
        rolling_downside_deviation_df[
            series_name
        ].to_numpy(dtype=float),
        label=series_name,
        color=risk_color_mapping[series_name],
        linewidth=2.0
    )

axes[0].set_title(
    "Rolling 36-Month Annualized Volatility",
    fontsize=14
)

axes[0].set_ylabel(
    "Annualized Volatility"
)

axes[0].yaxis.set_major_formatter(
    PercentFormatter(1.0)
)

axes[0].legend(
    frameon=False,
    loc="best"
)

axes[1].set_title(
    "Rolling 36-Month Downside Deviation",
    fontsize=14
)

axes[1].set_xlabel("Date")
axes[1].set_ylabel(
    "Annualized Downside Deviation"
)

axes[1].yaxis.set_major_formatter(
    PercentFormatter(1.0)
)

for axis in axes:
    axis.grid(alpha=0.25)

fig.tight_layout()

ROLLING_RISK_FIGURE = (
    RISK_FIGURE_DIR
    / "01_rolling_volatility_and_downside_risk.png"
)

fig.savefig(
    ROLLING_RISK_FIGURE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Rolling-risk figure was saved successfully.")

In [ ]:
var_lookback_window = 60
var_confidence_level = 0.95
var_tail_probability = (
    1.0 - var_confidence_level
)

rolling_var_records = []

for series_name in risk_series_names:
    series_returns = (
        risk_indexed_df[series_name]
        .dropna()
        .astype(float)
    )

    for position in range(
        var_lookback_window,
        len(series_returns)
    ):
        estimation_sample = (
            series_returns.iloc[
                position
                - var_lookback_window:
                position
            ]
        )

        forecast_date = (
            series_returns.index[position]
        )

        actual_return = (
            series_returns.iloc[position]
        )

        return_threshold = (
            estimation_sample.quantile(
                var_tail_probability
            )
        )

        tail_sample = estimation_sample[
            estimation_sample
            <= return_threshold
        ]

        var_forecast = -return_threshold
        cvar_forecast = -tail_sample.mean()

        var_exception = (
            actual_return
            < return_threshold
        )

        rolling_var_records.append({
            "series":
                series_name,

            "forecast_date":
                forecast_date,

            "actual_return":
                actual_return,

            "return_threshold":
                return_threshold,

            "var_95_forecast":
                var_forecast,

            "cvar_95_forecast":
                cvar_forecast,

            "var_exception":
                int(var_exception)
        })

rolling_var_backtest_df = pd.DataFrame(
    rolling_var_records
)

print("Rolling historical VaR forecasts were created successfully.")
print(
    "Number of forecasts:",
    len(rolling_var_backtest_df)
)

print("\nForecasts per series:")
print(
    rolling_var_backtest_df
    .groupby("series")
    .size()
)

In [ ]:
from scipy.stats import chi2


def calculate_kupiec_test(
    exception_series,
    expected_probability=0.05
):
    exceptions = (
        pd.Series(exception_series)
        .astype(int)
    )

    number_of_observations = len(exceptions)
    number_of_exceptions = int(
        exceptions.sum()
    )

    observed_probability = (
        number_of_exceptions
        / number_of_observations
    )

    null_log_likelihood = (
        (
            number_of_observations
            - number_of_exceptions
        )
        * np.log(
            1.0 - expected_probability
        )
        + number_of_exceptions
        * np.log(
            expected_probability
        )
    )

    if number_of_exceptions == 0:
        alternative_log_likelihood = 0.0

    elif (
        number_of_exceptions
        == number_of_observations
    ):
        alternative_log_likelihood = 0.0

    else:
        alternative_log_likelihood = (
            (
                number_of_observations
                - number_of_exceptions
            )
            * np.log(
                1.0 - observed_probability
            )
            + number_of_exceptions
            * np.log(
                observed_probability
            )
        )

    likelihood_ratio_statistic = (
        -2.0
        * (
            null_log_likelihood
            - alternative_log_likelihood
        )
    )

    p_value = chi2.sf(
        likelihood_ratio_statistic,
        df=1
    )

    return {
        "number_of_forecasts":
            number_of_observations,

        "expected_exceptions":
            (
                number_of_observations
                * expected_probability
            ),

        "observed_exceptions":
            number_of_exceptions,

        "observed_exception_rate":
            observed_probability,

        "kupiec_lr_statistic":
            likelihood_ratio_statistic,

        "kupiec_p_value":
            p_value,

        "reject_correct_coverage_at_5_percent":
            p_value < 0.05
    }


var_backtest_summary_records = []

for series_name, group in (
    rolling_var_backtest_df
    .groupby("series")
):
    kupiec_result = calculate_kupiec_test(
        exception_series=group[
            "var_exception"
        ],
        expected_probability=0.05
    )

    exception_sample = group.loc[
        group["var_exception"] == 1
    ]

    if len(exception_sample) > 0:
        realized_average_tail_loss = (
            -exception_sample[
                "actual_return"
            ].mean()
        )

        average_cvar_forecast_during_exceptions = (
            exception_sample[
                "cvar_95_forecast"
            ].mean()
        )
    else:
        realized_average_tail_loss = np.nan

        average_cvar_forecast_during_exceptions = (
            np.nan
        )

    kupiec_result.update({
        "series":
            series_name,

        "realized_average_tail_loss":
            realized_average_tail_loss,

        "average_cvar_forecast_during_exceptions":
            average_cvar_forecast_during_exceptions
    })

    var_backtest_summary_records.append(
        kupiec_result
    )

var_backtest_summary_df = (
    pd.DataFrame(
        var_backtest_summary_records
    )
    .set_index("series")
)

print("Historical VaR backtest summary:")
display(
    var_backtest_summary_df.round(4)
)

In [ ]:
strategy_var_plot_names = [
    "Quality Top Quintile",
    "Six-Factor Top Quintile"
]

fig, axes = plt.subplots(
    nrows=2,
    ncols=1,
    figsize=(12, 10),
    sharex=True
)

for axis, series_name in zip(
    axes,
    strategy_var_plot_names
):
    plot_sample = (
        rolling_var_backtest_df.loc[
            rolling_var_backtest_df[
                "series"
            ]
            == series_name
        ]
        .sort_values("forecast_date")
    )

    plot_dates = (
        plot_sample["forecast_date"]
        .to_numpy()
    )

    axis.plot(
        plot_dates,
        plot_sample[
            "actual_return"
        ].to_numpy(dtype=float),
        label="Actual Monthly Return",
        color="#1f77b4",
        linewidth=1.5
    )

    axis.plot(
        plot_dates,
        plot_sample[
            "return_threshold"
        ].to_numpy(dtype=float),
        label="Historical 95% VaR Threshold",
        color="#d62728",
        linewidth=1.8,
        linestyle="--"
    )

    exception_sample = plot_sample.loc[
        plot_sample["var_exception"] == 1
    ]

    axis.scatter(
        exception_sample[
            "forecast_date"
        ].to_numpy(),
        exception_sample[
            "actual_return"
        ].to_numpy(dtype=float),
        color="black",
        marker="x",
        s=70,
        label="VaR Exception",
        zorder=5
    )

    axis.axhline(
        y=0,
        color="gray",
        linewidth=0.8
    )

    axis.set_title(series_name)
    axis.set_ylabel("Monthly Return")

    axis.yaxis.set_major_formatter(
        PercentFormatter(1.0)
    )

    axis.legend(
        frameon=False,
        loc="lower right"
    )

    axis.grid(alpha=0.25)

axes[-1].set_xlabel("Date")

fig.suptitle(
    "Rolling Historical 95% VaR Backtest",
    fontsize=15,
    y=1.01
)

fig.tight_layout()

VAR_BACKTEST_FIGURE = (
    RISK_FIGURE_DIR
    / "02_rolling_historical_var_backtest.png"
)

fig.savefig(
    VAR_BACKTEST_FIGURE,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("VaR-backtest figure was saved successfully.")

In [ ]:
ROLLING_VAR_FILE = (
    PROCESSED_DATA_DIR
    / "28_rolling_historical_var_forecasts.csv"
)

VAR_BACKTEST_SUMMARY_FILE = (
    PROCESSED_DATA_DIR
    / "29_var_backtest_summary_2015_2025.csv"
)

ROLLING_VOLATILITY_FILE = (
    PROCESSED_DATA_DIR
    / "30_rolling_volatility_2015_2025.csv"
)

ROLLING_DOWNSIDE_FILE = (
    PROCESSED_DATA_DIR
    / "31_rolling_downside_deviation_2015_2025.csv"
)

rolling_var_backtest_df.to_csv(
    ROLLING_VAR_FILE,
    index=False
)

var_backtest_summary_df.to_csv(
    VAR_BACKTEST_SUMMARY_FILE
)

rolling_volatility_df.to_csv(
    ROLLING_VOLATILITY_FILE
)

rolling_downside_deviation_df.to_csv(
    ROLLING_DOWNSIDE_FILE
)

print("Rolling risk-analysis files were saved successfully.")

In [ ]:
def safe_log_probability(
    count,
    probability
):
    if count == 0:
        return 0.0

    if probability <= 0.0:
        return -np.inf

    return (
        count
        * np.log(probability)
    )


def calculate_christoffersen_test(
    exception_series
):
    exceptions = (
        pd.Series(exception_series)
        .astype(int)
        .to_numpy()
    )

    previous_exceptions = exceptions[:-1]
    current_exceptions = exceptions[1:]

    n00 = int(
        np.sum(
            (previous_exceptions == 0)
            & (current_exceptions == 0)
        )
    )

    n01 = int(
        np.sum(
            (previous_exceptions == 0)
            & (current_exceptions == 1)
        )
    )

    n10 = int(
        np.sum(
            (previous_exceptions == 1)
            & (current_exceptions == 0)
        )
    )

    n11 = int(
        np.sum(
            (previous_exceptions == 1)
            & (current_exceptions == 1)
        )
    )

    probability_after_no_exception = (
        n01 / (n00 + n01)
        if (n00 + n01) > 0
        else 0.0
    )

    probability_after_exception = (
        n11 / (n10 + n11)
        if (n10 + n11) > 0
        else 0.0
    )

    total_transitions = (
        n00 + n01 + n10 + n11
    )

    unconditional_transition_probability = (
        (n01 + n11)
        / total_transitions
    )

    null_log_likelihood = (
        safe_log_probability(
            n00 + n10,
            1.0
            - unconditional_transition_probability
        )
        + safe_log_probability(
            n01 + n11,
            unconditional_transition_probability
        )
    )

    alternative_log_likelihood = (
        safe_log_probability(
            n00,
            1.0
            - probability_after_no_exception
        )
        + safe_log_probability(
            n01,
            probability_after_no_exception
        )
        + safe_log_probability(
            n10,
            1.0
            - probability_after_exception
        )
        + safe_log_probability(
            n11,
            probability_after_exception
        )
    )

    independence_lr_statistic = (
        -2.0
        * (
            null_log_likelihood
            - alternative_log_likelihood
        )
    )

    independence_p_value = chi2.sf(
        independence_lr_statistic,
        df=1
    )

    return {
        "n00":
            n00,

        "n01":
            n01,

        "n10":
            n10,

        "n11":
            n11,

        "exception_probability_after_no_exception":
            probability_after_no_exception,

        "exception_probability_after_exception":
            probability_after_exception,

        "independence_lr_statistic":
            independence_lr_statistic,

        "independence_p_value":
            independence_p_value,

        "reject_exception_independence_at_5_percent":
            independence_p_value < 0.05
    }


conditional_coverage_records = []

for series_name, group in (
    rolling_var_backtest_df
    .sort_values("forecast_date")
    .groupby("series")
):
    independence_result = (
        calculate_christoffersen_test(
            group["var_exception"]
        )
    )

    kupiec_lr_statistic = (
        var_backtest_summary_df.loc[
            series_name,
            "kupiec_lr_statistic"
        ]
    )

    conditional_coverage_lr_statistic = (
        kupiec_lr_statistic
        + independence_result[
            "independence_lr_statistic"
        ]
    )

    conditional_coverage_p_value = chi2.sf(
        conditional_coverage_lr_statistic,
        df=2
    )

    independence_result.update({
        "series":
            series_name,

        "conditional_coverage_lr_statistic":
            conditional_coverage_lr_statistic,

        "conditional_coverage_p_value":
            conditional_coverage_p_value,

        "reject_correct_conditional_coverage_at_5_percent":
            conditional_coverage_p_value < 0.05
    })

    conditional_coverage_records.append(
        independence_result
    )

conditional_coverage_summary_df = (
    pd.DataFrame(
        conditional_coverage_records
    )
    .set_index("series")
)

print("Christoffersen conditional-coverage summary:")
display(
    conditional_coverage_summary_df.round(4)
)

In [ ]:
CONDITIONAL_COVERAGE_FILE = (
    PROCESSED_DATA_DIR
    / "32_christoffersen_conditional_coverage_test.csv"
)

conditional_coverage_summary_df.to_csv(
    CONDITIONAL_COVERAGE_FILE
)

print("Conditional-coverage results were saved successfully.")

## Risk Analysis Conclusions

The risk analysis provides several complementary findings.

First, the Quality Top Quintile strategy exhibited lower downside
deviation, historical CVaR, and maximum drawdown than the
equal-weighted S&P 500 benchmark. Its maximum drawdown was 25.20%,
compared with 27.47% for the equal-weighted benchmark, and it recovered
from the 2020 drawdown within eight months rather than eleven months.

Second, the capitalization-weighted S&P 500 remained the strongest
overall risk benchmark. It produced the lowest full-sample volatility,
downside deviation, and tail-loss measures. Therefore, quality screening
improved risk characteristics relative to an equal-weighted universe,
but did not dominate the capitalization-weighted market portfolio.

Third, the six-factor strategy illustrates why volatility alone is an
incomplete risk measure. Although its annualized volatility was
relatively moderate, it exhibited the deepest maximum drawdown, the
most negative skewness, the highest excess kurtosis, and the largest
99% conditional value at risk. Its risk was concentrated in relatively
infrequent but severe downside events.

The stress-period analysis also demonstrates substantial regime
dependence. The Quality strategy performed poorly during the 2015–2016
and 2018 selloffs, but offered meaningful downside protection during
the 2020 and 2022 market declines. The six-factor strategy performed
well during the 2022 decline but suffered the largest loss during the
2020 COVID-19 crash.

Finally, the rolling historical 95% VaR model generated four exceptions
over 72 out-of-sample forecasts for each return series, corresponding
to an exception rate of 5.56%. Neither the Kupiec unconditional-coverage
test nor the Christoffersen conditional-coverage test rejected the VaR
model at the 5% significance level. However, realized losses during VaR
exceptions were moderately larger than the corresponding CVaR
forecasts, particularly for the six-factor strategy. These results
should be interpreted cautiously because the monthly backtest contains
only 72 out-of-sample observations.

Overall, quality screening improved several downside-risk measures
relative to equal weighting, but the evidence does not support universal
risk dominance. Portfolio risk remained highly dependent on market
regime, benchmark choice, and the tail-risk measure used.